# Preparing text data for the LLM



### WORKING WITH TEXT DATA


In [1]:
import os #working with files
import urllib.request #downloading things from the internet
import nltk #library for NLP

In [2]:
with open("verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

In [3]:
len(raw_text)

20478

## TOKENIZATION

### USING Regular expression

In [4]:
import re
text="Hello , world . this is a test"
result=re.split(r"(\s)",text)
print(result)

['Hello', ' ', ',', ' ', 'world', ' ', '.', ' ', 'this', ' ', 'is', ' ', 'a', ' ', 'test']


In [5]:
# Splitting on punctuation, double hyphens, or whitespace using capturing groups
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)

# Remove any empty tokens or tokens that are just whitespace
preprocessed = [item.strip() for item in preprocessed if item.strip()]
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

## USING NLTK

In [ ]:
from nltk.tokenize import word_tokenize
# Replaces the preprocessed/regex logic completely inside encode():
preprocessed1 = word_tokenize(raw_text)
ids = [item for item in preprocessed1]
print(ids)

# CONVERTING TOKENS INTO TOKEN IDS

In [8]:
all_words=sorted(set(preprocessed)) ##REMOVES ALL DUPLICATE WORDS 
vocab_size=len(all_words)
print(vocab_size)

1130


In [9]:
vocab={token:integer for integer,token in enumerate(all_words)} 
print(vocab)

{'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29, 'Devonshire': 30, 'Don': 31, 'Dubarry': 32, 'Emperors': 33, 'Florence': 34, 'For': 35, 'Gallery': 36, 'Gideon': 37, 'Gisburn': 38, 'Gisburns': 39, 'Grafton': 40, 'Greek': 41, 'Grindle': 42, 'Grindles': 43, 'HAD': 44, 'Had': 45, 'Hang': 46, 'Has': 47, 'He': 48, 'Her': 49, 'Hermia': 50, 'His': 51, 'How': 52, 'I': 53, 'If': 54, 'In': 55, 'It': 56, 'Jack': 57, 'Jove': 58, 'Just': 59, 'Lord': 60, 'Made': 61, 'Miss': 62, 'Money': 63, 'Monte': 64, 'Moon-dancers': 65, 'Mr': 66, 'Mrs': 67, 'My': 68, 'Never': 69, 'No': 70, 'Now': 71, 'Nutley': 72, 'Of': 73, 'Oh': 74, 'On': 75, 'Once': 76, 'Only': 77, 'Or': 78, 'Perhaps': 79, 'Poor': 80, 'Professional': 81, 'Renaissance': 82, 'Ri

## NORMAL CODE

In [16]:
import re

class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}
        
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

In [21]:
tokenizer=SimpleTokenizerV1(vocab)
text="You are a painter , you paint good pride" ##Using words that are inside VOCAB
ids=tokenizer.encode(text)
print(ids)

[113, 169, 115, 747, 5, 1126, 745, 500, 793]


In [22]:
tokens=tokenizer.decode(ids)
print(tokens)

You are a painter, you paint good pride


# ADDING SPECIAL CONTEXT TOKENS

In [23]:
all_tokens=sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab={token:integer for integer,token in enumerate(all_tokens)}

In [28]:
len(vocab.items())
for i,item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [29]:
import re
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text


In [30]:
tokenizer=SimpleTokenizerV1(vocab)
text="You are a painter , you paint good pride hello sambar" ##Using any words
ids=tokenizer.encode(text)
print(ids)

[113, 169, 115, 747, 5, 1126, 745, 500, 793, 1131, 1131]


In [31]:
tokenizer.decode(tokenizer.encode(text))

'You are a painter, you paint good pride <|unk|> <|unk|>'

# BYTE PAIR ENCODING [used for handling unknown words]

In [35]:
import tiktoken
tiktoken.__version__

'0.13.0'

In [36]:
tokenizer=tiktoken.get_encoding("gpt2")
tokenizer.encode("hello world")

[31373, 995]

In [37]:
text=("Well!--even through the prism of Hermia's tears I felt able to face the fact with equanimity. Poor Jack Gisburn! The women had made him--it was fitting that they should mourn him. Among his own sex fewer regrets were heard, and in his own trade hardly a murmur. Professional jealousy? Perhaps. If it were, the honour of the craft was vindicated by little Claude Nutley")
tokenizer.encode(text,allowed_special={"<|endoftext|>"}) #endoftext is used to tell the LLM about a doc is ending , it is placed at end or start of a doc

[5779,
 28112,
 10197,
 832,
 262,
 46475,
 286,
 18113,
 544,
 338,
 10953,
 314,
 2936,
 1498,
 284,
 1986,
 262,
 1109,
 351,
 1602,
 11227,
 414,
 13,
 23676,
 3619,
 402,
 271,
 10899,
 0,
 383,
 1466,
 550,
 925,
 683,
 438,
 270,
 373,
 15830,
 326,
 484,
 815,
 25722,
 683,
 13,
 9754,
 465,
 898,
 1714,
 7380,
 30090,
 547,
 2982,
 11,
 290,
 287,
 465,
 898,
 3292,
 8941,
 257,
 4636,
 28582,
 13,
 18612,
 35394,
 30,
 8673,
 13,
 1002,
 340,
 547,
 11,
 262,
 15393,
 286,
 262,
 5977,
 373,
 29178,
 3474,
 416,
 1310,
 40559,
 11959,
 1636]

## DATA SAMPLING WITH SLIDING WINDOW

In [39]:
enc_text=tokenizer.encode(raw_text)
print(len(enc_text))

5144


In [44]:
enc_sample=enc_text[50:]
context_size=4
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]
print(f"x:{x}")
print(f"     y:{y}")

x:[290, 4920, 2241, 287]
     y:[4920, 2241, 287, 257]


In [46]:
import torch
torch.__version__
 

'2.13.0+cpu'

In [47]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        
        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
            
    def __len__(self):
        return len(self.input_ids)
        
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [48]:
import tiktoken
from torch.utils.data import DataLoader

def create_dataloader_v1(txt, batch_size=2, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")
    
    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    
    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    
    return dataloader


In [55]:
dataloader=create_dataloader_v1(
    raw_text,batch_size=1,max_length=4,stride=4,shuffle=False
)

data_iter=iter(dataloader)
first_batch=next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [54]:
dataloader=create_dataloader_v1(
    raw_text,batch_size=8,max_length=4,stride=4,shuffle=False
)

data_iter=iter(dataloader)
first_batch=next(data_iter)
print(first_batch)

[tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]), tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])]


# CREATING TOKEN EMBEDDINGS 

In [66]:
input_ids=torch.tensor([2,3,5,1])

vocab_size=6
output_dim=3
torch.manual_seed(42)
embedding_layer=torch.nn.Embedding(tokenizer.n_vocab,output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 1.9269,  1.4873,  0.9007],
        [-2.1055,  0.6784, -1.2345],
        [-0.0431, -1.6047, -0.7521],
        ...,
        [ 0.2875,  1.1914,  0.4735],
        [ 0.3121,  2.0992, -0.9541],
        [-0.0232,  0.4189,  0.7707]], requires_grad=True)


In [67]:
embedding_layer(torch.tensor([3]))

tensor([[ 1.6487, -0.3925, -1.4036]], grad_fn=<EmbeddingBackward0>)

In [68]:
embedding_layer(input_ids)

tensor([[-0.0431, -1.6047, -0.7521],
        [ 1.6487, -0.3925, -1.4036],
        [ 0.7624,  1.6423, -0.1596],
        [-2.1055,  0.6784, -1.2345]], grad_fn=<EmbeddingBackward0>)

## POSITIONAL ENCODING

In [70]:
vocab_size=50257
output_dim=256
token_embedding_layer=torch.nn.Embedding(vocab_size,output_dim)

In [71]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)


In [72]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)


Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [73]:
token_embeddings=token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

## ADDING POSITONAL ENCODINGS

In [79]:
context_len=max_length
pos_emb_layer=torch.nn.Embedding(context_len,output_dim)
print(torch.arange(max_length))

tensor([0, 1, 2, 3])


In [80]:
pos_emb_layer.weight

Parameter containing:
tensor([[-1.1432,  0.2496,  0.3955,  ...,  0.6589, -0.6985, -0.3505],
        [ 0.4345,  0.5863,  0.0883,  ...,  0.1670, -0.9053,  0.1349],
        [ 0.7763,  0.0238, -0.5792,  ..., -1.3416, -0.1418,  0.0381],
        [ 0.3229, -0.1072, -0.1188,  ...,  0.0901, -0.1477, -0.9448]],
       requires_grad=True)

In [81]:
pos_emb_layer(torch.arange(max_length))

tensor([[-1.1432,  0.2496,  0.3955,  ...,  0.6589, -0.6985, -0.3505],
        [ 0.4345,  0.5863,  0.0883,  ...,  0.1670, -0.9053,  0.1349],
        [ 0.7763,  0.0238, -0.5792,  ..., -1.3416, -0.1418,  0.0381],
        [ 0.3229, -0.1072, -0.1188,  ...,  0.0901, -0.1477, -0.9448]],
       grad_fn=<EmbeddingBackward0>)

In [82]:
pos_emb=pos_emb_layer(torch.arange(max_length))
print(pos_emb.shape)

torch.Size([4, 256])


In [83]:
print(token_embeddings.shape)

torch.Size([8, 4, 256])
